In [1]:
import random
import json
import pickle

import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Dropout
from tensorflow.keras.optimizers import SGD

import numpy as np

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
lemmatizer = WordNetLemmatizer()

intents = json.loads(open("intents.json").read())

words = []
classes = []
documents = []

ignore_letters = ["?", "!", ".", ","]

for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)
        documents.append((word_list, intent["tag"]))

        if intent["tag"] not in classes:
            classes.append(intent["tag"])
words = [lemmatizer.lemmatize(word.lower())
         for word in words if word not in ignore_letters]

words = sorted(set(words))
classes = sorted(set(classes))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\amreen\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\amreen\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\amreen\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\amreen\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
pickle.dump(words, open('words.pkl', 'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))

In [4]:
dataset = []
template = [0]*len(classes)

for document in documents:
    bag = []
    word_patterns = document[0]
    word_patterns = [lemmatizer.lemmatize(
        word.lower()) for word in word_patterns]

    for word in words:
        bag.append(1) if word in word_patterns else bag.append(0)

    output_row = list(template)
    output_row[classes.index(document[1])] = 1
    dataset.append([bag, output_row])

random.shuffle(dataset)
dataset = np.array(dataset, dtype=object)

train_x = list(dataset[:, 0])
train_y = list(dataset[:, 1])

In [5]:
model = Sequential()
model.add(Dense(256, input_shape=(len(train_x[0]),),
                activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))


sgd = SGD(learning_rate=0.01, decay=1e-6,
          momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy',
              optimizer=sgd, metrics=['accuracy'])

hist = model.fit(np.array(train_x), np.array(train_y),
                 epochs=200, batch_size=5, verbose=1)

model.save("chatbot_model.h5", hist)
print("Done!")

C:\Users\amreen\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200


C:\Users\amreen\anaconda3\Lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.0130 - loss: 4.3693
Epoch 2/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0130 - loss: 4.3429
Epoch 3/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0208 - loss: 4.3348
Epoch 4/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0156 - loss: 4.3116
Epoch 5/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0468 - loss: 4.2847
Epoch 6/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0442 - loss: 4.2646
Epoch 7/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0338 - loss: 4.2338
Epoch 8/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0701 - loss: 4.1813
Epoch 9/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0857 - loss: 4.1412
Epoch 10/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0805 - loss: 4.0912
Epoch 11/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1013 - loss: 3.9790
Epoch 12/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1351 - lo

Done!


In [6]:
import random
import json
import pickle
import numpy as np
import nltk

from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

# ------------------ DOWNLOAD REQUIRED DATA ------------------
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

# ------------------ LOAD DATA ------------------
intents = json.loads(open("intents.json").read())

words = []
classes = []
documents = []

ignore_letters = ["?", "!", ".", ","]

# ------------------ PROCESS DATA ------------------
for intent in intents["intents"]:
    for pattern in intent["patterns"]:
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)
        documents.append((word_list, intent["tag"]))

        if intent["tag"] not in classes:
            classes.append(intent["tag"])

# Lemmatize and clean
words = [lemmatizer.lemmatize(word.lower())
         for word in words if word not in ignore_letters]

words = sorted(set(words))
classes = sorted(set(classes))

# Save vocabulary
pickle.dump(words, open('words.pkl', 'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))

# ------------------ CREATE TRAINING DATA ------------------
training = []
output_empty = [0] * len(classes)

for document in documents:
    bag = []
    word_patterns = document[0]
    word_patterns = [lemmatizer.lemmatize(word.lower())
                     for word in word_patterns]

    for word in words:
        bag.append(1 if word in word_patterns else 0)

    output_row = list(output_empty)
    output_row[classes.index(document[1])] = 1

    training.append([bag, output_row])

# Shuffle data
random.shuffle(training)

# Split X and Y (IMPORTANT FIX 🔥)
train_x = []
train_y = []

for item in training:
    train_x.append(item[0])
    train_y.append(item[1])

# Convert to numpy arrays
train_x = np.array(train_x)
train_y = np.array(train_y)

# ------------------ BUILD MODEL ------------------
model = Sequential()

model.add(Dense(256, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(len(train_y[0]), activation='softmax'))

# ------------------ COMPILE ------------------
sgd = SGD(learning_rate=0.01, momentum=0.9, nesterov=True)

model.compile(loss='categorical_crossentropy',
              optimizer=sgd,
              metrics=['accuracy'])

# ------------------ TRAIN ------------------
hist = model.fit(train_x, train_y,
                 epochs=200,
                 batch_size=5,
                 verbose=1)

# ------------------ SAVE MODEL ------------------
model.save("chatbot_model.h5")

print("✅ Model trained and saved successfully!")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\amreen\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\amreen\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\amreen\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\amreen\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Epoch 1/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.0156 - loss: 4.3568
Epoch 2/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0208 - loss: 4.3488
Epoch 3/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0156 - loss: 4.3334
Epoch 4/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0364 - loss: 4.3121
Epoch 5/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0312 - loss: 4.2873
Epoch 6/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0312 - loss: 4.2858
Epoch 7/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0649 - loss: 4.2321
Epoch 8/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0649 - loss: 4.2018
Epoch 9/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0831 - loss: 4.1392
Epoch 10/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0805 - loss: 4.0712
Epoch 11/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1091 - loss: 4.0042
Epoch 12/200
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy:

✅ Model trained and saved successfully!


In [7]:
# This function will take the voice input
# converted into string as input and predict
# and return the result in both
# text as well as voice format

def calling_the_bot(txt):
    global res
    predict = predict_class(txt)
    res = get_response(predict, intents)

    engine.say("Found it. From our Database we found that" + res)
    # engine.say(res)
    engine.runAndWait()
    print("Your Symptom was  : ", text)
    print("Result found in our Database : ", res)

In [8]:
!pip install SpeechRecognition pyttsx3 pyaudio

In [9]:
import speech_recognition as sr
import pyttsx3
import time


In [10]:
import pickle
from tensorflow.keras.models import load_model

# Load model
model = load_model("chatbot_model.h5")

words = pickle.load(open("words.pkl", "rb"))
classes = pickle.load(open("classes.pkl", "rb"))

# ------------------ FUNCTIONS ------------------
def clean_up_sentence(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word.lower())
                      for word in sentence_words]
    return sentence_words

def bag_of_words(sentence):
    sentence_words = clean_up_sentence(sentence)
    bag = [0]*len(words)

    for w in sentence_words:
        for i, word in enumerate(words):
            if word == w:
                bag[i] = 1

    return np.array(bag)

def predict_class(sentence):
    bow = bag_of_words(sentence)
    res = model.predict(np.array([bow]), verbose=0)[0]

    ERROR_THRESHOLD = 0.1

    results = [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)

    return_list = []
    for r in results:
        return_list.append({
            "intent": classes[r[0]],
            "probability": str(r[1])
        })

    return return_list

def get_response(intents_list, intents_json):
    if len(intents_list) == 0:
        return "Sorry, I couldn't understand your symptoms."

    tag = intents_list[0]['intent']

    for intent in intents_json['intents']:
        if intent['tag'] == tag:
            return random.choice(intent['responses'])

In [11]:
import speech_recognition as sr
import pyttsx3
import time

def calling_the_bot(txt, engine):
    predict = predict_class(txt)
    res = get_response(predict, intents)
    
    print("\n--- Diagnostic Result ---")
    print("Your Symptom was :", txt)
    print("Diagnosis        :", res)
    print("-------------------------\n")
    
    engine.say(res)
    engine.runAndWait()

print("Starting voice engine...")
recognizer = sr.Recognizer()
mic = sr.Microphone()

engine = pyttsx3.init()
engine.setProperty('rate', 170)
engine.setProperty('volume', 1.0)
voices = engine.getProperty('voices')

engine.say("Hello user, I am your healthcare chatbot. Please tell me your symptoms.")
engine.runAndWait()

network_error_count = 0 

while True:
    text_input = None
    
    # If internet fails multiple times, switch to text mode
    if network_error_count >= 2:
        print("\n[Voice is disabled due to network issues]")
        text_input = input("Please type your symptoms (or type 'exit' to quit): ")
        if text_input.lower() in ["exit", "stop", "quit"]:
            print("Bot stopped.")
            engine.say("Shutting down. Take care!")
            engine.runAndWait()
            break
        calling_the_bot(text_input, engine)
        continue
        
    print("\nListening for your symptoms...")
    try:
        with mic as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            # Give maximum 5 seconds to speak
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=10)
            
        print("Processing voice...")
        text_input = recognizer.recognize_google(audio)
        print("You said:", text_input)
        
        network_error_count = 0 # Reset errors on success

        if text_input.lower() in ["exit", "no", "stop", "quit"]:
            print("Bot stopped")
            engine.say("Shutting down. Take care!")
            engine.runAndWait()
            break

        calling_the_bot(text_input, engine)

    except sr.UnknownValueError:
        print("Could not understand your voice. Please try again.")
        engine.say("Sorry, I didn't catch that. Could you repeat?")
        engine.runAndWait()
        
    except sr.RequestError as e:
        print(f"\n[Network Error]: Cannot reach Google Speech Service. Error detail: {e}")
        network_error_count += 1
        # Fallback to Text!
        text_input = input("Voice recognition failed. Please type your symptoms (or 'exit' to stop): ")
        
        if text_input.lower() in ["exit", "stop", "quit"]:
            print("Bot stopped")
            break
            
        calling_the_bot(text_input, engine)
        
    except Exception as e:
        # Ignore random microphone timeouts gracefully
        pass


Starting voice engine...

Listening for your symptoms...
Processing voice...
You said: I have high fever and joint pain

--- Diagnostic Result ---
Your Symptom was : I have high fever and joint pain
Diagnosis        : Possible dengue.
-------------------------


Listening for your symptoms...

Listening for your symptoms...
Processing voice...
You said: exit
Bot stopped
